# ESM3 design retention — what it writes for a backbone

ESM3 is the one masked model here that conditions on a backbone, so unlike ESMC
it can be asked what it writes when it redesigns a chain. Only the
`struct_cond` arm can: with the structure track withheld there is nothing to
redesign against, and generation would be an unconditional draw.

Kept apart from `esm3_scoring.ipynb` deliberately. That notebook is a 2x2 that
finishes in about fifteen minutes; this one is measured in hours, and bolting
it on would mean a runtime death costs the scoring too.


## 1. GPU, dependencies, gated checkpoint

In [ ]:
!nvidia-smi -L


In [ ]:
# Verified on a clean Python 3.13 environment before being written here,
# because Colab runs 3.13 while ARC runs 3.12 and the two need different
# recipes. Three things this gets right that earlier versions did not:
#
#   esm==3.2.2 CANNOT install on Colab. It declares requires_python
#   >=3.12,<3.13, so the pin copied from the ARC setup is unsatisfiable here.
#   3.4.0 is the version that supports 3.13.
#
#   --no-deps on `esm` protects torch. esm declares torch<2.12,>=2.11, and
#   letting pip resolve that would replace Colab's CUDA-matched build --
#   worse than any missing package. Its OTHER dependencies then install
#   normally, because transformers validates its own deps at import and
#   --no-deps everywhere leaves it broken.
#
#   Not quiet, and verified by importing in a FRESH interpreter, so neither a
#   silent pip failure nor a stale import in this kernel can look like success.
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps',
                'esm==3.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers>=4.57.6,<5.0.0', 'einops', 'biotite>=1.0.0',
                'msgpack-numpy', 'biopython', 'scikit-learn', 'brotli',
                'attrs', 'pandas', 'cloudpathlib', 'httpx', 'tenacity',
                'zstd', 'huggingface_hub', 'safetensors', 'pygtrie',
                'accelerate', 'ipython'], check=True)

# pip will warn that esm wants torch<2.12 and rdkit. Both are expected: torch
# is deliberately left alone, and rdkit is not on this code path.
probe = subprocess.run([sys.executable, '-c', '''
import torch, esm
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein
from esm.utils.structure.protein_chain import ProteinChain
from esm.tokenization import EsmSequenceTokenizer
t = EsmSequenceTokenizer()
assert t.vocab_size == 33 and t.mask_token_id == 32
assert t.convert_tokens_to_ids("N") == 17
print(f"esm {esm.__version__} | torch {torch.__version__} | vocab OK")
'''], capture_output=True, text=True)
print(probe.stdout.strip() or probe.stderr.strip()[-2500:])
assert probe.returncode == 0, (
    'esm did not install usably. If the error names a package already imported\n'
    'by this kernel, use Runtime -> Restart session and run this cell FIRST.')


In [ ]:
# esm3-sm-open-v1 is gated: accept the licence at
# https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1 first, then paste a token.
# Run this only after the one-time restart requested by the install cell.
# This import catches an in-memory mix of the old and newly installed Hub.
from huggingface_hub.utils import refresh_xet_connection_info
from huggingface_hub import login
login()


## 2. Repository and data

In [ ]:
import os, subprocess, sys
from pathlib import Path

BRANCH = 'fix/context-extractor-mapping'
REPO   = 'https://github.com/LBDillon/Glycan-occupancy-analysis.git'
BUNDLE = ('https://github.com/LBDillon/Glycan-occupancy-analysis/releases/'
          'download/bundle-2026-08-20/colab_bundle.tar')
MODULE = '/content/module'

# Fetch and reset rather than skipping when the directory exists. Restarting a
# Colab session restarts the kernel but keeps /content, so a checkout from an
# earlier run survives -- and a clone guarded only by directory existence then
# silently keeps stale code, which reads as 'no module named ...' for anything
# added since.
if os.path.exists(MODULE):
    subprocess.run(['git', '-C', MODULE, 'fetch', '--depth', '1', 'origin', BRANCH],
                   check=True)
    subprocess.run(['git', '-C', MODULE, 'reset', '--hard', f'origin/{BRANCH}'],
                   check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, MODULE],
                   check=True)

head = subprocess.run(['git', '-C', MODULE, 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout.strip()
print('HEAD:', head)

# The whole point of this notebook is code that only exists on this branch, so
# check it arrived rather than discovering it three cells later.
required = ['src/experimental_glycosylation_sites/adapters/esm3.py',
            'src/experimental_glycosylation_sites/esm3_scoring.py']
missing = [f for f in required if not os.path.exists(f'{MODULE}/{f}')]
assert not missing, (f'{missing} absent from the checkout. The branch may not be '
                     'pushed, or this is a stale clone: rm -rf /content/module '
                     'and rerun.')
print('esm3 adapter present')
assert not head.startswith('aeff61d'), (
    'This is the stale pre-autocast checkout that failed on a T4. Rerun '
    'this cell after the branch has been pushed; do not patch the smoke '
    'model by hand because the full run starts a fresh process.')


In [ ]:
# Plain Python rather than shell magic: `find -exec` and IPython's brace
# substitution disagree about escaping, and a half-extracted bundle looks
# exactly like a complete one.
import gzip, shutil, tarfile, urllib.request

STRUCT = Path(MODULE) / 'data/cache/pdb'
STRUCT.mkdir(parents=True, exist_ok=True)

if not any(STRUCT.iterdir()):
    archive_path = Path('/content/colab_bundle.tar')
    if not archive_path.exists():
        print('downloading 594 MB...', flush=True)
        urllib.request.urlretrieve(BUNDLE, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall('/content/extracted')

    source = Path('/content/extracted')
    for path in (source / 'structures').rglob('*'):
        if path.suffix == '.gz':
            with gzip.open(path) as fh, open(STRUCT / path.stem, 'wb') as out:
                shutil.copyfileobj(fh, out)
        elif path.suffix in ('.pdb', '.cif'):
            shutil.copy(path, STRUCT / path.name)

    for kind in ('manifests', 'matching'):
        target = Path(MODULE) / 'results' / kind
        target.mkdir(parents=True, exist_ok=True)
        for path in (source / kind).glob('*.csv'):
            shutil.copy(path, target / path.name)

structures = list(STRUCT.iterdir())
manifests = list((Path(MODULE) / 'results/manifests').glob('*.csv'))
print('structures:', len(structures), '(expect 1824)')
print('manifests :', len(manifests))
assert structures and manifests, 'bundle did not extract; rerun this cell'


## 3. Verify design before running anything long

The same three chains the scorer was verified on, so anything that fails here
is about generation rather than about parsing.

In [ ]:
import sys
sys.path.insert(0, f'{MODULE}/src')
os.chdir(MODULE)

from experimental_glycosylation_sites.adapters.esm3 import ESM3Adapter
from experimental_glycosylation_sites.esm3_scoring import checked_chain

adapter = ESM3Adapter(device='cuda')
model, _ = adapter._load()

# One design at a time leaves the GPU idle -- the first pilot measured ~13
# unmasking steps per second whatever the chain length, so cost was steps times
# designs and hardly touched by length. Batching fills the device. If this
# checkpoint has no batch_generate the run still works, and takes roughly the
# batch size longer.
print('batch_generate available:', hasattr(model, 'batch_generate'))
print('generation provenance   :', adapter.describe_generation())
CASES = [('4EBY', 'A', 27, 'NSS'), ('5H5Y', 'A', 226, 'NRS'),
         ('9G3Q', 'A', 181, 'NES')]

for pdb, chain, n, triplet in CASES:
    path = STRUCT / f'{pdb}.pdb'
    if not path.exists():
        print(f'{pdb}: not in the bundle, skipped'); continue
    native, _ = checked_chain(str(path), chain)
    assert native[n:n + 3] == triplet, (
        f'{pdb}: manifest index {n} reads {native[n:n+3]!r}, not {triplet!r}')

    designs = adapter.design(str(path), chain, n_designs=4, temperature=0.1)

    # Full length, or every index in the retention read-out is shifted.
    assert all(len(d) == len(native) for d in designs), 'a design changed length'
    at_sequon = sorted({d[n:n + 3] for d in designs})
    identity = sum(a == b for a, b in zip(designs[0], native)) / len(native)
    print(f'{pdb}:{chain}  L={len(native)}  steps={adapter.design_steps_for(len(native))}'
          f'  native {triplet}  designs at the sequon: {at_sequon}')
    print(f'         sequence identity to native, design 0: {identity:.1%}')
    assert len(set(designs)) > 1, f'{pdb}: every design identical'

# The sequon must be free to vanish, or retention measures the constraint.
print('\nNothing is fixed: the triplets above are whatever the model wrote.')

# And seq_only must refuse rather than draw unconditionally.
try:
    ESM3Adapter(device='cuda', structure_mode='seq_only').design(
        str(STRUCT / '4EBY.pdb'), 'A', n_designs=1, temperature=0.1)
    raise AssertionError('seq_only designed without a backbone')
except ValueError as exc:
    print('seq_only refused, as it must:', str(exc)[:70])


## 4. What to design, and what it will cost

The occupied cases live in the 2640-site scoring manifest and the controls in
their own, so a paired retention result needs both arms — but only the cases a
matched pair actually names. That is 194 chains rather than 1725.

The estimate before this section existed was five to eight hours, extrapolated
from the scorer's timings rather than from generation. The pilot below replaces
it with a measurement.

In [ ]:
import numpy as np
import pandas as pd

KEY = ['accession', 'position', 'structure_pdb_id', 'structure_chain_id']
CH = ['structure_pdb_id', 'structure_chain_id']
N_DESIGNS, PILOT_CHAINS = 32, 20

pairs = pd.read_csv('results/matching/matched_pairs_secretory.csv', low_memory=False)
big = pd.read_csv('results/manifests/scoring_manifest.csv', low_memory=False)
if 'scoreable' in big.columns:
    big = big[big.scoreable.astype(bool)]
big['accession'] = big.accession.astype(str)
big['position'] = big.position.astype(int)
wanted = set(zip(pairs.case_accession.astype(str), pairs.case_position.astype(int)))
cases = big[[(a, p) in wanted for a, p in zip(big.accession, big.position)]]
controls = pd.read_csv('results/manifests/manifest_matched_secretory.csv',
                       low_memory=False)

all_chains = pd.concat([cases[CH], controls[CH]], ignore_index=True).drop_duplicates()
print(f'{len(cases.drop_duplicates(KEY))} case sites on {len(cases.drop_duplicates(CH))} chains')
print(f'{len(controls.drop_duplicates(KEY))} control sites on {len(controls.drop_duplicates(CH))} chains')
print(f'{len(all_chains)} chains to design in total\n')


### The pilot

Twenty chains, sampled across the length range. A few minutes, and it replaces a guess at the cost with a measurement.

The first run of this pilot measured 10.0 h for 427 chains generating one design at a time. Batching the designs is what that number bought.


In [ ]:
import time

# Sampled across the length range rather than off the top: chains are ordered by
# PDB id, and generation cost scales with length, so the first twenty would not
# be representative of the rest.
rng = np.random.default_rng(0)
sample = all_chains.iloc[rng.permutation(len(all_chains))[:PILOT_CHAINS]]

timings, lengths = [], []
for row in sample.itertuples(index=False):
    path = STRUCT / f'{row.structure_pdb_id}.pdb'
    if not path.exists():
        continue
    try:
        native, _ = checked_chain(str(path), row.structure_chain_id)
    except Exception as exc:
        print(f'  {row.structure_pdb_id}/{row.structure_chain_id}: '
              f'{type(exc).__name__}, skipped'); continue
    t0 = time.time()
    adapter.design(str(path), row.structure_chain_id,
                   n_designs=N_DESIGNS, temperature=0.1)
    elapsed = time.time() - t0
    timings.append(elapsed); lengths.append(len(native))
    print(f'  {row.structure_pdb_id}/{row.structure_chain_id}  L={len(native):5d}  '
          f'steps={adapter.design_steps_for(len(native)):3d}  {elapsed:6.1f}s  '
          f'({elapsed/N_DESIGNS:.2f}s/design)')

per_chain = float(np.mean(timings))
total_h = per_chain * len(all_chains) / 3600
print(f'\n{len(timings)} chains timed, '
      f'batched={adapter.describe_generation()["batched"]}')
print(f'  mean {per_chain:.1f}s/chain, median length {int(np.median(lengths))}')
print(f'  MEASURED estimate for {len(all_chains)} chains: {total_h:.1f} h')
print(f'  a Colab session is ~12 h, and the run resumes, so that is '
      f'{max(1, round(total_h / 10)):.0f} session(s)')


## 5. Drive, and the manifest to design

Mounted before the long run, not after it. Anything already in Drive is copied
back, so a fresh runtime resumes rather than restarting — stage 08 skips sites
already present in its output.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/glyco_occupancy/esm3_design')
OUT.mkdir(parents=True, exist_ok=True)

designs_dir = Path(MODULE) / 'results/designs'
analysis_dir = Path(MODULE) / 'results/analysis'
designs_dir.mkdir(parents=True, exist_ok=True)
analysis_dir.mkdir(parents=True, exist_ok=True)


def usable(path):
    """Whether a checkpoint holds work stage 08 can actually resume.

    A run killed before its first flush leaves a zero-byte file; older builds
    then appended to it without a header, so the rows are there but unlabelled
    and nothing can tell which sites they cover. Restoring such a file only
    makes the stage refuse it.
    """
    if path.stat().st_size == 0:
        return False, 'empty'
    if path.name.startswith('retention_') and 'accession' not in path.open().readline():
        return False, 'headerless'
    return True, ''


discarded = []
for path in sorted(designs_dir.glob('*.csv')) + sorted(analysis_dir.glob('*.csv')):
    ok, why = usable(path)
    if not ok:
        path.unlink(); discarded.append(f'{path.name} ({why})')
if discarded:
    print('discarded from the runtime, no resumable work in them:')
    for name in discarded:
        print('  -', name)

restored, skipped = 0, []
for path in OUT.glob('*'):
    if not path.is_file():
        continue
    target = analysis_dir if path.suffix == '.json' or path.name.startswith(
        ('retention_paired', 'retention_by_class')) else designs_dir
    if path.suffix == '.csv':
        ok, why = usable(path)
        if not ok:
            skipped.append(f'{path.name} ({why})'); continue
    if not (target / path.name).exists():
        shutil.copy(path, target / path.name); restored += 1
if skipped:
    print('NOT restored, no resumable work in them:', ', '.join(skipped))
print(f'Drive: restored {restored} files into the runtime')


def save_to_drive(note=''):
    """Copy results out. Called after each arm, not only at the end."""
    copied = 0
    for pattern in ('results/designs/retention_*esm3*.csv',
                    'results/analysis/retention_*esm3*.json'):
        for path in Path(MODULE).glob(pattern):
            shutil.copy(path, OUT / path.name); copied += 1
    print(f'  saved {copied} files to Drive {note}'.rstrip())


In [ ]:
# Stated because stage 10 will still write them: ESM3's CLASS AVERAGES come from
# this matched subset and are NOT comparable with ProteinMPNN's, which come from
# the full manifest. The paired result from stage 10b is the usable output.
CASE_MANIFEST = Path(MODULE) / 'results/manifests/manifest_esm3_design_cases.csv'
cases.to_csv(CASE_MANIFEST, index=False)
print(f'wrote {CASE_MANIFEST.name}: {len(cases.drop_duplicates(KEY))} sites, '
      f'{len(cases.drop_duplicates(CH))} chains')

ARMS = [
    ('scoring_manifest', CASE_MANIFEST,
     Path(MODULE) / 'results/designs/retention_scoring_manifest_esm3.csv'),
    ('secretory', Path(MODULE) / 'results/manifests/manifest_matched_secretory.csv',
     Path(MODULE) / 'results/designs/retention_secretory_esm3.csv'),
]
for tag, manifest, out in ARMS:
    print(f'  {tag:18} {manifest.name} -> {out.name}')


## 6. The design run

Resumable: stage 08 skips sites already in its output, and Drive is written
after each arm. If the runtime dies, rerun from the top — the install and clone
cells are idempotent and this cell picks up where it stopped.

In [ ]:
import subprocess, time


def run_designs(tag, manifest, out):
    """One arm. Shows what the child said, rather than a bare CalledProcessError."""
    print(f'\n=== {tag} ===', flush=True)
    started = time.time()
    done = subprocess.run(
        [sys.executable, '-u', 'pipeline/08_design.py', str(manifest), str(out),
         '--model', 'esm3', '--device', 'auto'],
        capture_output=True, text=True, cwd=MODULE)
    print(done.stdout[-4000:] or '(no stdout)')
    if done.returncode:
        print('--- STDERR ---')
        print(done.stderr[-4000:] or '(no stderr)')
        raise SystemExit(f'{tag} exited {done.returncode}; the cause is above')
    print(f'elapsed {(time.time() - started) / 60:.0f} min')
    save_to_drive(f'after {tag}')


for tag, manifest, out in ARMS:
    run_designs(tag, manifest, out)


## 7. Validate coverage before trusting anything

In [ ]:
problems = []
for tag, manifest, out in ARMS:
    if not out.exists():
        print(f'{tag}: not run'); continue
    designed = pd.read_csv(out, low_memory=False)
    fail_path = out.with_name(out.stem + '_failures.csv')
    failures = pd.DataFrame(columns=KEY)
    if fail_path.exists() and fail_path.stat().st_size > 1:
        try:
            failures = pd.read_csv(fail_path, low_memory=False)
        except pd.errors.EmptyDataError:
            pass

    source = pd.read_csv(manifest, low_memory=False)
    if 'scoreable' in source.columns:
        source = source[source.scoreable.astype(bool)]
    expected = len(source.drop_duplicates(KEY))

    parts = [designed.reindex(columns=KEY)]
    if len(failures):
        parts.append(failures.reindex(columns=KEY))
    accounted = len(pd.concat(parts, ignore_index=True).drop_duplicates(KEY))
    short = expected - accounted
    flag = '' if short <= 0 else f'  <-- {short} NEVER ATTEMPTED'
    print(f'{tag}: {len(designed)} designed + {len(failures)} failed '
          f'= {accounted} of {expected}{flag}')
    if short > 0:
        problems.append(f'{tag}: {short} unaccounted')

    if 'generation' in designed.columns:
        kinds = set(designed.generation.dropna())
        print(f'    generation recorded as: {kinds}')
        assert kinds <= {'masked_diffusion_unmasking'}, (
            f'{tag}: rows labelled {kinds}; a masked diffusion model must not be '
            'recorded as autoregressive, or these cannot be told apart later')

if problems:
    raise SystemExit('rerun the design cell; it resumes and will fill these in: '
                     + '; '.join(problems))
print('\nevery site accounted for')


## 8. Retention

In [ ]:
for stage in ('10_analyse_retention_by_class.py',
              '10b_analyse_retention_paired.py'):
    print(f'=== {stage} ===', flush=True)
    done = subprocess.run([sys.executable, '-u', f'pipeline/{stage}',
                           '--variant', 'esm3'],
                          capture_output=True, text=True, cwd=MODULE)
    print(done.stdout[-3000:] or '(no stdout)')
    if done.returncode:
        print('--- STDERR ---'); print(done.stderr[-3000:])
        raise SystemExit(f'{stage} exited {done.returncode}')
save_to_drive('final')

result = json.loads(
    (Path(MODULE) / 'results/analysis/retention_paired_esm3.json').read_text())
secretory = result['comparisons'].get('eukaryotic secretory')
print('\nESM3, eukaryotic secretory, paired retention:')
for key in ('n_pairs', 'occupied_mean', 'control_mean', 'paired_difference',
            'ci95', 'n_tied', 'wilcoxon_p'):
    if key in secretory:
        print(f'  {key:20} {secretory[key]}')
print('\nMany pairs are usually tied -- both sites lose the sequon -- so the'
      '\neffect rests on far fewer informative pairs than n_pairs suggests.')


## 9. Download everything to the laptop

Self-contained: safe to run on its own in a fresh runtime.


In [ ]:
# Standalone: needs no other cell to have run in this session. Takes the Drive
# copy of every ESM3 design artefact, plus anything newer sitting in the runtime
# that the last save_to_drive did not catch, and lays them out the way the repo
# expects so the archive unzips into the project root.
import zipfile
from pathlib import Path

from google.colab import drive, files

try:
    drive.mount('/content/drive')
except Exception as exc:                      # already mounted is not an error
    print('mount:', str(exc)[:80])

DRIVE = Path('/content/drive/MyDrive/glyco_occupancy/esm3_design')
MODULE = Path('/content/module')


def route(name):
    """Where the pipeline looks for this file, which is not where Drive keeps it."""
    if name.startswith(('retention_paired', 'retention_by_class')):
        return 'results/analysis'
    return 'results/designs'


# Drive first, then the runtime -- and the runtime wins on a tie only if it is
# newer, so a stage that ran after the last save is still collected.
found = {}
if DRIVE.is_dir():
    for path in DRIVE.iterdir():
        if path.is_file():
            found[path.name] = path
for pattern in ('results/designs/retention_*esm3*.csv',
                'results/analysis/retention_*esm3*.json',
                'results/analysis/retention_*esm3*.csv'):
    for path in MODULE.glob(pattern):
        if (path.name not in found
                or path.stat().st_mtime > found[path.name].stat().st_mtime):
            found[path.name] = path

archive = Path('/content/esm3_design_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name, path in sorted(found.items()):
        bundle.write(path, f'{route(name)}/{name}')

print(f'{len(found)} files, {archive.stat().st_size / 1e6:.2f} MB\n')
for name, path in sorted(found.items()):
    where = 'drive' if str(path).startswith('/content/drive') else 'runtime'
    print(f'  {route(name):16} {name:48} {path.stat().st_size:>9,} B  ({where})')

# Named explicitly so a gap is visible here rather than as a puzzle later.
EXPECTED = {
    'retention_scoring_manifest_esm3.csv': 'the occupied cases',
    'retention_secretory_esm3.csv': 'the matched controls',
    'retention_all_classes_esm3.csv': 'stage 10',
    'retention_by_class_esm3.json': 'stage 10',
    'retention_paired_esm3.json': 'stage 10b -- the result',
}
missing = {n: why for n, why in EXPECTED.items() if n not in found}
if missing:
    print('\nMISSING:')
    for name, why in missing.items():
        print(f'  {name:48} ({why})')
    print('  run the retention cell first if the stage-10 files are absent')
else:
    print('\neverything expected is present')

files.download(str(archive))
